# 01 · Ingesta y Exploración
## Segmentación Inteligente de Pacientes — Programa de Medicina Funcional · Comfama

| | |
|---|---|
| **Propósito** | Cargar el histórico real de encuestas + datos clínicos y evaluar su calidad antes de modelar. |
| **Entrada** | `data/HISTORICO_MF_HACKATON2026 (1).xlsx`, hoja `Datos` (16.204 filas × 20 columnas, 4.388 pacientes únicos — en promedio ~3.7 consultas por paciente). |
| **Salida** | `data/_processed/01_raw_clean.csv` (capa *bronze*: columnas renombradas, tipos numéricos corregidos). |
| **Reemplaza a** | La versión anterior de este notebook usaba `Preguntas y respuestas medico.xlsx` (bloque de columnas L:AB, ~4.170 registros, casi exclusivamente cohorte Cardiometabólico). Esta versión usa el histórico completo `HISTORICO_MF_HACKATON2026`, con las 19 variables de estilo de vida **y** mediciones clínicas (IMC, perímetro abdominal, presión arterial, colesterol, HbA1c) en una sola hoja — no requiere aislar un bloque de columnas ni asumir una cohorte predominante. |

> ⚠️ Notebook adaptado para ejecutarse en **Jupyter local** (pandas/scikit-learn puro, sin `dbutils`/Spark/Delta) para poder validar el pipeline de punta a punta. La lógica es igualmente válida en Databricks — basta con reemplazar las celdas de lectura/escritura local por `dbutils.widgets` y tablas de Unity Catalog si se despliega allí.

### Paso 1 · Parámetros

In [1]:
from pathlib import Path

INPUT_PATH = Path("../data/HISTORICO_MF_HACKATON2026 (1).xlsx")
SHEET_NAME = "Datos"
PROCESSED_DIR = Path("../data/_processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### Paso 2 · Mapa de columnas (trazabilidad Excel → modelo)
Se renombra por posición (más robusto que por nombre, dado lo largas y variables que son las preguntas originales del formulario).

In [2]:
import pandas as pd

RAW_COLUMNS = [
    "documento", "habitos_alim", "actividad_tipo", "actividad_duracion",
    "actividad_frecuencia", "estres_alto", "ansioso", "nervioso",
    "tecnica_estres", "medicamentos", "perimetro_abdominal", "peso", "talla",
    "imc", "pa_sistolica", "pa_diastolica", "sintomas_gi",
    "colesterol_total", "colesterol_hdl", "hba1c",
]

df = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)
df.columns = RAW_COLUMNS
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]} | Pacientes únicos: {df['documento'].nunique():,}")
df.head(3)


Filas: 16,204 | Columnas: 20 | Pacientes únicos: 4,388


,documento,habitos_alim,actividad_tipo,actividad_duracion,actividad_frecuencia,estres_alto,ansioso,nervioso,tecnica_estres,medicamentos,perimetro_abdominal,peso,talla,imc,pa_sistolica,pa_diastolica,sintomas_gi,colesterol_total,colesterol_hdl,hba1c
0,43041805,Ninguno;,"Cardiovascular (caminar, trotar, spinning, bic...",Más de 30 minutos,Ninguno;,No,Sí,Sí,No,Ninguno,113,86,1.65,31.588613,118,78,Ninguno;,152,47,5.9
1,42983595,Incorporar más vegetales en su alimentación;Au...,Ninguno;,Ninguno;,Ninguno;,No,No,No,No,Hipolipemiantes;,100,74,1.59,29.270994,118,78,Ninguno;,199,55,5.6
2,49655828,Ninguno;,"Cardiovascular (caminar, trotar, spinning, bic...",Más de 30 minutos,Ninguno;,No,Sí,No,No,Ninguno,99,78,1.51,34.209026,118,78,Ninguno;,200,52,5.9


### Paso 3 · Tipos de datos
`perimetro_abdominal`, `peso`, `talla` y `pa_diastolica` llegan como texto porque algunos valores usan coma decimal (formato es-CO, p. ej. `78,5`). Se convierten a float.

In [3]:
def to_float_co(series):
    return pd.to_numeric(series.astype(str).str.strip().str.replace(",", ".", regex=False), errors="coerce")

df["perimetro_abdominal"] = to_float_co(df["perimetro_abdominal"])
df["pa_diastolica"] = to_float_co(df["pa_diastolica"])
# 'peso' y 'talla' no se usan como feature (el IMC ya viene calculado en el Excel),
# se dejan solo como referencia.
df["peso"] = to_float_co(df["peso"])
df["talla"] = to_float_co(df["talla"])

print(df.dtypes)


documento                 int64
habitos_alim             object
actividad_tipo           object
actividad_duracion       object
actividad_frecuencia     object
estres_alto              object
ansioso                  object
nervioso                 object
tecnica_estres           object
medicamentos             object
perimetro_abdominal     float64
peso                    float64
talla                   float64
imc                     float64
pa_sistolica              int64
pa_diastolica           float64
sintomas_gi              object
colesterol_total          int64
colesterol_hdl            int64
hba1c                   float64
dtype: object


### Paso 4 · Nulos

In [4]:
print(df.isna().sum())


documento               0
habitos_alim            0
actividad_tipo          0
actividad_duracion      0
actividad_frecuencia    0
estres_alto             0
ansioso                 0
nervioso                0
tecnica_estres          0
medicamentos            0
perimetro_abdominal     0
peso                    0
talla                   0
imc                     0
pa_sistolica            0
pa_diastolica           5
sintomas_gi             0
colesterol_total        0
colesterol_hdl          0
hba1c                   0
dtype: int64


### Paso 5 · Outliers fisiológicamente imposibles
Antes de modelar, se cuantifican valores que no pueden ser correctos (errores de captura manual).

In [5]:
checks = {
    "IMC fuera de [10, 70]": ~df["imc"].between(10, 70),
    "Perímetro abdominal fuera de [40, 180] cm": ~df["perimetro_abdominal"].between(40, 180),
    "PA sistólica fuera de [70, 220] mmHg": ~df["pa_sistolica"].between(70, 220),
    "PA diastólica fuera de [40, 150] mmHg": ~df["pa_diastolica"].between(40, 150),
    "HbA1c fuera de [3, 16] %": ~df["hba1c"].between(3, 16),
}
for label, mask in checks.items():
    print(f"{label}: {int(mask.sum())} registros")

mask_valid = ~pd.concat(checks.values(), axis=1).any(axis=1)
print(f"\nTotal a descartar: {(~mask_valid).sum()} de {len(df)} ({(~mask_valid).mean()*100:.2f}%)")


IMC fuera de [10, 70]: 4 registros
Perímetro abdominal fuera de [40, 180] cm: 13 registros
PA sistólica fuera de [70, 220] mmHg: 7 registros
PA diastólica fuera de [40, 150] mmHg: 12 registros
HbA1c fuera de [3, 16] %: 0 registros

Total a descartar: 30 de 16204 (0.19%)


### Paso 6 · Distribución de variables categóricas clave

In [6]:
for col in ["actividad_tipo", "actividad_duracion", "actividad_frecuencia",
            "estres_alto", "ansioso", "nervioso", "tecnica_estres"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts(dropna=False).head(8))



=== actividad_tipo ===
actividad_tipo
Cardiovascular (caminar, trotar, spinning, bicicleta, elíptica, natación, baile, aeróbicos);    7792
Ninguno;                                                                                        6145
Cardiovascular más fuerza;                                                                      2130
Fuerza (pesas);                                                                                  137
Name: count, dtype: int64

=== actividad_duracion ===
actividad_duracion
Más de 30 minutos       7594
Ninguno;                6145
Menos de 30 minutos     1996
Menos de 30 minutos      375
 Más de 30 minutos        94
Name: count, dtype: int64

=== actividad_frecuencia ===
actividad_frecuencia
Ninguno;         15030
3 veces            475
4 veces            261
5 o más veces      218
2 veces            145
1 vez               46
2 veces             29
Name: count, dtype: int64

=== estres_alto ===
estres_alto
No    13506
Sí     2698
Name: count, dtype

In [7]:
print("=== medicamentos ===")
print(df["medicamentos"].value_counts().head(10))
print("\n=== síntomas gastrointestinales ===")
print(df["sintomas_gi"].value_counts().head(10))


=== medicamentos ===
medicamentos
Ninguno                                                                                                9880
Hipolipemiantes;                                                                                       2344
Hipoglicemiantes; Hipolipemiantes                                                                      1786
Inhibidores de la bomba de protones (Esomeprazol, lanzoprazol, pantoprazol, omeprazol);                 900
Hipoglicemiantes;                                                                                       521
Antiácidos ;Inhibidores de la bomba de protones (Esomeprazol, lanzoprazol, pantoprazol, omeprazol);     385
Antiácidos;                                                                                             253
Antiácidos;Inhibidores de la bomba de protones (Esomeprazol, lanzoprazol, pantoprazol, omeprazol);      110
Hipolipemiantes                                                                                       

### Paso 7 · Estadística descriptiva de variables clínicas numéricas

In [8]:
df[mask_valid][["imc", "perimetro_abdominal", "pa_sistolica", "pa_diastolica",
                "colesterol_total", "colesterol_hdl", "hba1c"]].describe().round(2)


,imc,perimetro_abdominal,pa_sistolica,pa_diastolica,colesterol_total,colesterol_hdl,hba1c
count,16174.00,16174.00,16174.00,16174.00,16174.00,16174.00,16174.00
mean,28.39,90.81,118.34,77.01,193.05,49.25,5.59
std,5.70,13.73,7.87,5.62,40.51,11.76,0.59
min,14.44,41.00,84.00,50.00,67.00,19.00,4.00
25%,24.34,81.00,118.00,76.00,167.00,41.00,5.30
50%,27.48,90.00,118.00,78.00,192.00,48.00,5.50
75%,31.85,99.00,120.00,80.00,215.00,55.00,5.70
max,65.57,175.00,190.00,120.00,423.00,133.00,14.40


### Paso 8 · Persistencia (capa *bronze*)
Se guardan los registros válidos con tipos corregidos. Punto de partida versionado para el notebook 02.

In [9]:
df_clean = df[mask_valid].reset_index(drop=True)
out_path = PROCESSED_DIR / "01_raw_clean.csv"
df_clean.to_csv(out_path, index=False)
print(f"Guardado: {out_path} ({len(df_clean):,} filas)")


Guardado: ..\data\_processed\01_raw_clean.csv (16,174 filas)


---
**Resumen del notebook 01:** histórico cargado, tipado y depurado de outliers
fisiológicamente imposibles (~30 de 16.204 filas, <0.2%). Sin problemas
relevantes de nulos.
➡️ Continúa en **`02_preprocesamiento`** para la ingeniería de features.